In [14]:
import sys, importlib
sys.path.insert(0, '..')

# Reload everything fresh
import src.auto_mapper, src.extractor, src.metric_builder
import src.calculation_engine, src.vector_pipeline, src.answer_engine
import pipeline

for mod in [src.auto_mapper, src.extractor, src.metric_builder,
            src.calculation_engine, src.vector_pipeline, src.answer_engine, pipeline]:
    importlib.reload(mod)

from pipeline import run_full_pipeline, ask
print("Ready.")


Ready.


In [15]:
# ============================================================
# CHANGE THESE 3 LINES FOR EACH NEW COMPANY
# ============================================================
COMPANY  = "tesla"
PDF_PATH = "../data/Tesla/tesla 22 10-k.pdf"
YEAR     = 2022
# ============================================================
print(f"Company: {COMPANY.upper()} | Year: {YEAR}")


Company: TESLA | Year: 2022


In [16]:
result = run_full_pipeline(
    company_name = COMPANY,
    pdf_path     = PDF_PATH,
    year         = YEAR,
)



  FULL PIPELINE: TESLA  |  FY2022

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [49, 80]
  Auto-detected balance sheet pages    : [48]
  Combined target pages: [48, 49, 80]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: tesla.json

[Step 3/5] Extracting metrics from PDF...

  Company : TESLA
  PDF     : tesla 22 10-k.pdf
  Year    : 2022  |  Columns: [2022, 2021, 2020]
  Pages   : [48, 49, 80]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: tesla 22 10-k.pdf (251 pages total)
  Extracted 66 candidate financial lines (74 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 66 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Tesla\tesla_2022_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 66 rows for 'tesla'...
  [exact]  'cash and cash equiv

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9641.22it/s]


  Created new collection: 'tesla_2022'
  Chunking PDF text...
  Opened PDF: tesla 22 10-k.pdf (251 pages total)
  -> 2076 text chunks from PDF
  Converting metrics to text sentences...
  -> 27 metric fact sentences
  -> 2103 total chunks to embed
  Embedding 2103 chunks in batches of 50...
  Stored 50/2103 chunks...
  Stored 100/2103 chunks...
  Stored 150/2103 chunks...
  Stored 200/2103 chunks...
  Stored 250/2103 chunks...
  Stored 300/2103 chunks...
  Stored 350/2103 chunks...
  Stored 400/2103 chunks...
  Stored 450/2103 chunks...
  Stored 500/2103 chunks...
  Stored 550/2103 chunks...
  Stored 600/2103 chunks...
  Stored 650/2103 chunks...
  Stored 700/2103 chunks...
  Stored 750/2103 chunks...
  Stored 800/2103 chunks...
  Stored 850/2103 chunks...
  Stored 900/2103 chunks...
  Stored 950/2103 chunks...
  Stored 1000/2103 chunks...
  Stored 1050/2103 chunks...
  Stored 1100/2103 chunks...
  Stored 1150/2103 chunks...
  Stored 1200/2103 chunks...
  Stored 1250/2103 chunks...
  St

In [18]:
questions = [
    f"What was {COMPANY.title()}'s gross margin in {YEAR}?",
    f"What was the net profit margin in {YEAR}?",
    f"What was the revenue growth in {YEAR}?",
    f"What was the R&D as a percentage of revenue?",
]
print("DETERMINISTIC ANSWERS")
print("=" * 50)
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q, result, use_llm=False)}")


DETERMINISTIC ANSWERS

Q: What was Tesla's gross margin in 2022?
A: Gross margin percentage in 2022 was 25.60%.

Q: What was the net profit margin in 2022?
A: Net profit margin in 2022 was 15.45%.

Q: What was the revenue growth in 2022?
A: Total net sales changed by 51.35% in 2022 compared with 2021.

Q: What was the R&D as a percentage of revenue?
A: R&D expense as a percentage of sales in 2022 was 3.77%.


In [19]:
rag_questions = [
    f"What are the main products of {COMPANY.title()}?",
    f"What risks does {COMPANY.title()} mention?",
    f"What is {COMPANY.title()}'s strategy for growth?",
    f"what is {COMPANY.title()}'s profit in year 2022 in percentages and amount too?",
]
print("RAG + LLM ANSWERS")
print("=" * 50)
for q in rag_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q, result)}")
    print("-" * 40)


RAG + LLM ANSWERS

Q: What are the main products of Tesla?
A: The main products of Tesla, according to the provided context, are high-performance fully electric vehicles and energy generation and storage systems.
----------------------------------------

Q: What risks does Tesla mention?
A: Tesla mentions several risks in the FY2022 10-K annual report. These include:

1. Operating in a cyclical industry that is sensitive to political and regulatory uncertainty, including with respect to trade and the environment, which can be compounded by inflationary pressures, rising energy prices, increases in interest rates, and any future global impact from the COVID-19 pandemic (Source 1).
2. Managing risks related to planned high-volume product sales, market and geographical expansion, and technological innovations (Source 2).
3. Employee turnover due to a competitive labor market for talented individuals with automotive or technology experience, or any negative publicity related to Tesla (Sour

In [20]:
print(ask("What did the company say about future growth?", result))


The report states that the company is focused on growing their manufacturing capacity, which includes ramping all production vehicles to their installed production capacities, increasing production rate, efficiency, and capacity at current factories. The next phase of production growth will depend on the ramp at Gigafactory Berlin-Brandenburg and Gigafactory Texas, as well as their ability to add to their available sources of battery cell supply by manufacturing their own cells they are developing. Additionally, they aim to expand operations to enable increased deliveries and deployments of their products for further revenue growth.
